# Nano Video Generation: All-in-One Tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AshkanTaghipour/nano-video-gen/blob/main/notebooks/nano_video_gen_colab.ipynb)

A self-contained notebook that teaches the concepts behind modern video diffusion models (Wan 2.1 architecture) with a ~2.0M parameter **NanoDiT** model you can train and run inference on in minutes.

The NanoDiT uses the **real pretrained Wan 2.1 VAE** and **T5 text encoder** -- the same components used by the full-scale models -- so it operates on real latent representations and real text embeddings.

**What you will learn:**
1. How the VAE compresses video into a latent space
2. Every building block of the Diffusion Transformer (RoPE, RMSNorm, AdaIN, attention, FFN)
3. How the full DiT architecture assembles these blocks
4. The flow matching diffusion paradigm (training & inference)
5. Training a tiny video generation model on real video data with real VAE + T5
6. Generating videos with Classifier-Free Guidance (CFG)

**Runtime**: ~20 minutes end-to-end on a Colab T4 GPU (includes ~9.5 GB model download on first run).

---
## Setup

Clone the repository and install dependencies. This cell only needs to run once per Colab session.

In [ ]:
import os

# Clone the repo (or pull latest if already cloned)
if not os.path.exists('/content/nano-video-gen'):
    !git clone https://github.com/AshkanTaghipour/nano-video-gen.git /content/nano-video-gen
else:
    !cd /content/nano-video-gen && git pull

%cd /content/nano-video-gen

# Install dependencies (no conda on Colab)
!pip install -q torch torchvision einops matplotlib imageio imageio-ffmpeg Pillow tqdm \
    diffusers transformers accelerate sentencepiece huggingface_hub

In [ ]:
import sys
import torch
import gc

# Ensure the project root is on the Python path
sys.path.insert(0, '/content/nano-video-gen')

# Verify GPU
if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    device = torch.device('cuda')
else:
    print("No GPU detected -- running on CPU (training will be slower).")
    device = torch.device('cpu')

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

# Download pretrained Wan 2.1 VAE + T5 weights (~9.5 GB, cached after first run)
model_path = '/content/nano-video-gen/pretrained_models/Wan2.1'

from nano_video_gen.model.wan_vae_wrapper import WanVAEWrapper
from nano_video_gen.model.t5_text_encoder import T5TextEncoder, CachedTextEmbeddings

print("\nLoading Wan 2.1 VAE on CPU (downloads on first run)...")
vae = WanVAEWrapper(model_path=model_path, device="cpu")
print(f"WanVAE loaded (latent_channels={vae.latent_channels})")

# Download DiffSynth-Studio example video dataset (~27 MB)
# Real video clips with text captions from ModelScope
data_dir = '/content/nano-video-gen/data/example_video_dataset'

if not os.path.exists(data_dir):
    print("\nDownloading DiffSynth-Studio example video dataset...")
    !git lfs install
    !git clone https://www.modelscope.cn/datasets/DiffSynth-Studio/example_video_dataset.git {data_dir}
    %cd {data_dir}
    !git lfs pull
    %cd /content/nano-video-gen
    print("Done!")
else:
    print(f"\nDataset already exists at: {data_dir}")

# Verify dataset loads
from nano_video_gen.data.dataset import VideoDataset
_check = VideoDataset(data_dir, height=128, width=128, num_frames=17)
print(f"Dataset: {len(_check)} videos, {_check.num_prompts} unique prompts")
del _check

---
# Section 1: Video Generation Overview

Video generation synthesizes realistic video frames from a conditioning signal -- typically a text prompt. Adding the **temporal dimension** to image generation introduces major challenges:

| Aspect | Image Generation | Video Generation |
|--------|-----------------|------------------|
| Dimensionality | 2D (H x W) | 3D (T x H x W) |
| Consistency | Single frame | Must be coherent across many frames |
| Compute | Manageable | Grows linearly (or worse) with frame count |
| Motion | N/A | Must model physically plausible motion |

Modern video diffusion models like **Wan 2.1** tackle these challenges with three core components:

1. **Video VAE** -- compresses raw pixel-space video into a compact *latent* representation
2. **Text Encoder** -- converts a natural-language prompt into dense embeddings
3. **Diffusion Transformer (DiT)** -- iteratively *denoises* noise in latent space, guided by text

### Pipeline Flow

```
Text Prompt
    |  
    v
[Text Encoder] --> text embeddings (B, seq_len, text_dim)
    |  
    v
Random Noise z_T ~ N(0, I)  in latent space (B, C_lat, T', H', W')
    |  
    v  (iterative denoising, N steps)
[DiT Model]  x  text embeddings  x  timestep  -->  predicted velocity
    |  
    v  (Euler step: z_{t-1} = z_t + v * delta_sigma)
Clean latent z_0  (B, C_lat, T', H', W')
    |  
    v
[VAE Decoder]  -->  video pixels  (B, 3, T, H, W)
```

### Why Latent Diffusion? An Analogy

Imagine you want to write directions for painting the Mona Lisa, pixel by pixel. A 128×128 video with 17 frames has **786,432 values** — that's like writing directions for nearly a million individual paint dots!

**Latent diffusion** is like working with a *sketch* instead. The VAE compresses the video into a compact "sketch" (the latent), and the DiT model only needs to learn how to create good sketches (~20K values). The VAE decoder then "paints" the final video from the sketch.

This is the single most important idea in modern video generation:

> **Don't generate pixels directly. Generate a compressed representation, then decode it.**

Benefits:
- **40x fewer values** to denoise per step → much faster training and inference
- **Semantically meaningful** — the latent captures "what's in the scene", not individual pixel colors
- **Memory efficient** — fits on consumer GPUs instead of requiring massive clusters

In [ ]:
# Architecture comparison: Wan 14B vs Wan 1.3B vs Nano
#
# Our Nano model preserves every architectural concept
# from the full Wan model, just at a much smaller scale.
# It uses the SAME pretrained VAE and T5 text encoder.

header = f"{'Component':<20} {'Wan 14B':>10} {'Wan 1.3B':>10} {'Nano':>10}"
sep    = '-' * len(header)

rows = [
    ('DiT dim',           '5120',  '1536',  '128'),
    ('Attention heads',   '40',    '12',    '4'),
    ('Transformer layers','40',    '30',    '2'),
    ('FFN dim',           '13824', '8960',  '512'),
    ('Latent channels',   '16',    '16',    '16'),
    ('Text dim',          '4096',  '4096',  '4096'),
    ('Freq dim',          '256',   '256',   '64'),
    ('Patch size',        '[1,2,2]','[1,2,2]','[1,2,2]'),
    ('~Parameters',       '~14B',  '~1.3B', '~2.0M'),
]

print(header)
print(sep)
for name, wan14, wan1, nano in rows:
    print(f"{name:<20} {wan14:>10} {wan1:>10} {nano:>10}")

print()
print("Note: Nano uses the SAME pretrained Wan 2.1 VAE (16 latent ch)")
print("and T5 text encoder (4096-dim) -- only the DiT is tiny.")

### Why Not Just Downsample? VAE vs Simple Resizing

You might wonder: "Why use a complex VAE? Can't we just resize the video to be smaller?"

**Simple downsampling** (e.g., shrinking 128×128 → 16×16) throws away information destructively. You can never recover the original.

A **VAE** learns to compress *intelligently*:
- It keeps the most important information (edges, objects, motion patterns)
- It discards redundancy (large uniform regions, repeated textures)
- The 16 latent channels encode different *semantic features* (not just R, G, B)
- The decoder can reconstruct near-perfect video from these compact features

Think of it like the difference between:
- **JPEG compression** (smart, keeps what matters) vs **just deleting pixels** (dumb, loses everything)

The Wan 2.1 VAE was trained on millions of videos to learn this optimal compression. We use it frozen — no need to retrain it.

In [ ]:
# Compute the compression ratio of latent diffusion
# Using the real Wan 2.1 VAE dimensions

# Pixel-space video dimensions (128x128, 17 frames)
B, C, T, H, W = 1, 3, 17, 128, 128
pixel_values = C * T * H * W

# Latent-space dimensions after Wan VAE encoding
# Spatial: 8x compression, Temporal: (T-1)//4 + 1
C_lat = 16
T_lat = (T - 1) // 4 + 1  # 5
H_lat, W_lat = H // 8, W // 8  # 16, 16
latent_values = C_lat * T_lat * H_lat * W_lat

ratio = pixel_values / latent_values

print(f"Pixel space:  [{B}, {C}, {T}, {H}, {W}] = {pixel_values:,} values per sample")
print(f"Latent space: [{B}, {C_lat}, {T_lat}, {H_lat}, {W_lat}] = {latent_values:,} values per sample")
print(f"Compression ratio: {ratio:.0f}x")
print()
print("The DiT model only needs to denoise ~20K values")
print("instead of ~786K values per diffusion step!")

---
# Section 2: VAE & Latent Space

The **Video VAE** (Variational Autoencoder) compresses raw pixel-space video into a compact latent representation suitable for diffusion.

- **Encoder**: maps `[B, 3, T, H, W]` to a low-dimensional latent `[B, 16, T', H', W']`
- **Decoder**: reconstructs the video from the latent back to `[B, 3, T, H, W]`

We use the **pretrained Wan 2.1 VAE** (`AutoencoderKLWan`) which was already loaded in the setup cell above. It uses CausalConv3d layers and provides:
- **16 latent channels** (vs 3 RGB channels)
- **8x spatial compression** (128x128 -> 16x16)
- **4x temporal compression** with the `(T-1)//4 + 1` formula (17 frames -> 5 latent frames)

We will encode and decode a real video from the **synthetic moving-shapes dataset** (generated in the setup cell) to see the VAE in action.

In [ ]:
import matplotlib.pyplot as plt
from nano_video_gen.visualization import visualize_latent_space

# The pretrained Wan 2.1 VAE was loaded in the setup cell as `vae`
print(f"VAE type: {type(vae).__name__}")
print(f"Latent channels (z_dim): {vae.latent_channels}")
print(f"Device: {vae.device}")
print(f"\nNormalization: per-channel latents_mean + latents_std (16 values each)")
print(f"  latents_mean: {[f'{v:.2f}' for v in vae.latents_mean.flatten().tolist()]}")
print(f"  latents_std:  {[f'{v:.2f}' for v in vae.latents_std.flatten().tolist()]}")
print(f"\nThis is the same pretrained VAE used by Wan 14B and Wan 1.3B.")
print("It is frozen (no trainable parameters) and runs on CPU to save GPU VRAM.")

In [ ]:
# Encode -> Decode demo with the real Wan 2.1 VAE
# Load a real video from the synthetic dataset (generated in setup cell)
from nano_video_gen.data.dataset import VideoDataset

demo_dataset = VideoDataset(
    base_path=data_dir, height=128, width=128, num_frames=17,
)
sample = demo_dataset[0]
video = sample["video"].unsqueeze(0)  # [1, 3, 17, 128, 128]
print(f"Loaded video: \"{sample['prompt']}\"")
print(f"Input video shape:          {list(video.shape)}")
print(f"Input values per sample:    {video[0].numel():,}")

vae.eval()
with torch.no_grad():
    latent = vae.encode(video)
    reconstruction = vae.decode(latent)

print(f"Latent shape:               {list(latent.shape)}")
print(f"  - 16 latent channels (vs 3 RGB)")
print(f"  - {latent.shape[2]} temporal frames ((17-1)//4 + 1 = 5)")
print(f"  - {latent.shape[3]}x{latent.shape[4]} spatial (128//8 = 16)")
print(f"Compression ratio:          {video[0].numel() / latent[0].numel():.1f}x")
print(f"Reconstruction shape:       {list(reconstruction.shape)}")

In [ ]:
# Visualize all 16 latent channels from the Wan VAE
fig = visualize_latent_space(
    latent,
    title="Wan 2.1 VAE Latent Channels (first temporal frame)",
    figsize=(14, 6),
)
plt.show()

print(f"The Wan VAE produces {latent.shape[1]} latent channels.")
print("Each channel captures different aspects of the video content.")

In [ ]:
# Reconstruction quality of the pretrained Wan 2.1 VAE
with torch.no_grad():
    mse = torch.nn.functional.mse_loss(reconstruction, video)
    mae = torch.nn.functional.l1_loss(reconstruction, video)
    psnr = 10 * torch.log10(4.0 / mse)

print(f"=== Reconstruction Error (pretrained Wan 2.1 VAE) ===")
print(f"Video: \"{sample['prompt']}\"")
print(f"MSE:  {mse.item():.4f}")
print(f"MAE:  {mae.item():.4f}")
print(f"PSNR: {psnr.item():.2f} dB")
print()
print("The pretrained VAE achieves high-quality reconstruction of the synthetic video.")
print("On complex real-world video, the PSNR is typically > 30 dB.")

# Visualize original vs reconstruction (first frame)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

orig_frame = ((video[0, :, 0].permute(1, 2, 0).clamp(-1, 1) + 1) / 2).numpy()
recon_frame = ((reconstruction[0, :, 0].permute(1, 2, 0).clamp(-1, 1) + 1) / 2).numpy()

axes[0].imshow(orig_frame)
axes[0].set_title('Original (frame 0)')
axes[0].axis('off')

axes[1].imshow(recon_frame)
axes[1].set_title(f'Reconstruction (PSNR={psnr.item():.1f} dB)')
axes[1].axis('off')

plt.suptitle('Wan 2.1 VAE: Encode -> Decode', fontsize=14)
plt.tight_layout()
plt.show()

### What Do Latent Channels Actually Encode?

Unlike RGB channels (which simply store red, green, blue intensity), VAE latent channels encode **learned abstract features**. Let's explore what each channel "sees" by removing one channel at a time and observing the effect on the reconstructed video.

In [ ]:
# Interactive: Explore what happens when we zero out individual latent channels
# This reveals what each channel contributes to the final video
import numpy as np

with torch.no_grad():
    fig, axes = plt.subplots(2, 5, figsize=(16, 9))

    # Original reconstruction
    recon_orig = vae.decode(latent)
    orig_frame = ((recon_orig[0, :, 0].permute(1, 2, 0).clamp(-1, 1) + 1) / 2).numpy()
    axes[0, 0].imshow(orig_frame)
    axes[0, 0].set_title('Original\n(all 16 channels)', fontsize=9)
    axes[0, 0].axis('off')

    # Zero out one channel at a time and see what changes
    channels_to_test = [0, 2, 5, 8, 12]
    for i, ch in enumerate(channels_to_test):
        latent_modified = latent.clone()
        latent_modified[0, ch] = 0  # Zero out this channel
        recon_mod = vae.decode(latent_modified)
        mod_frame = ((recon_mod[0, :, 0].permute(1, 2, 0).clamp(-1, 1) + 1) / 2).numpy()

        # Show modified reconstruction
        axes[0, i].imshow(mod_frame) if i > 0 else None
        if i > 0:
            axes[0, i].set_title(f'Without ch {ch}', fontsize=9)
            axes[0, i].axis('off')

        # Show the difference (what that channel contributed)
        diff = np.abs(orig_frame - mod_frame)
        diff_normalized = diff / (diff.max() + 1e-8)
        axes[1, i].imshow(diff_normalized)
        axes[1, i].set_title(f'Ch {ch} contribution\n(brighter = more impact)', fontsize=9)
        axes[1, i].axis('off')

    plt.suptitle('What Each Latent Channel Contributes to the Video', fontsize=14)
    plt.subplots_adjust(hspace=0.35)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

print("Each channel encodes different aspects: edges, colors, textures, motion...")
print("This is why 16 channels can encode more information than 3 RGB channels!")

---
# Section 3: Building Blocks of the DiT

The Diffusion Transformer is built from a small set of reusable components. Let's examine each one.

## 3.1 Sinusoidal Embeddings

**The Problem:** The diffusion model receives a single number — the timestep $t$ (e.g., $t = 500$) — that tells it "how noisy is the input right now?" But a neural network can't do much with a single number. It needs a rich, high-dimensional representation.

**The Solution:** Encode the timestep using sine and cosine waves at different frequencies, just like a radio signal:
- **Low-frequency waves** (slowly varying) → distinguish "early denoising" from "late denoising"
- **High-frequency waves** (rapidly varying) → distinguish nearby timesteps ($t=500$ vs $t=501$)

**Why sine/cosine specifically?**
1. They're **smooth** — nearby timesteps get similar embeddings
2. They have **infinite range** — works for any timestep value
3. They're **unique** — no two timesteps share the same embedding
4. The same idea was first used for word positions in the original Transformer paper (Vaswani et al., 2017)

$$\text{embed}(t) = [\cos(t \cdot \omega_0), \cos(t \cdot \omega_1), \ldots, \sin(t \cdot \omega_0), \sin(t \cdot \omega_1), \ldots]$$

where $\omega_i = 10000^{-i/(d/2)}$ creates exponentially spaced frequencies.

In [ ]:
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

from nano_video_gen.model.components import (
    sinusoidal_embedding_1d,
    precompute_freqs_cis_3d,
    rope_apply,
    RMSNorm,
    modulate,
    GateModule,
)
from nano_video_gen.model.attention import SelfAttention, CrossAttention
from nano_video_gen.visualization import (
    visualize_rope_frequencies,
    visualize_modulation_effect,
    visualize_attention_maps,
    visualize_ffn_activations,
)

torch.manual_seed(42)

# Sinusoidal embeddings for various timesteps
dim = 64
timesteps = torch.tensor([0, 50, 100, 250, 500, 750, 1000], dtype=torch.float32)
embeddings = sinusoidal_embedding_1d(dim, timesteps)

print(f"Timesteps: {timesteps.tolist()}")
print(f"Embedding shape: {list(embeddings.shape)}  (num_timesteps x dim)")

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

im = axes[0].imshow(embeddings.numpy(), cmap='RdBu', aspect='auto', interpolation='nearest')
axes[0].set_xlabel('Embedding dimension')
axes[0].set_ylabel('Timestep')
axes[0].set_yticks(range(len(timesteps)))
axes[0].set_yticklabels([f't={int(t)}' for t in timesteps])
axes[0].set_title('Sinusoidal Timestep Embeddings')
plt.colorbar(im, ax=axes[0])

dense_t = torch.linspace(0, 1000, 200)
dense_emb = sinusoidal_embedding_1d(dim, dense_t)
for ch in [0, 8, 16, 31]:
    axes[1].plot(dense_t.numpy(), dense_emb[:, ch].numpy(), label=f'dim {ch}')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Embedding value')
axes[1].set_title('Individual frequency channels (low to high frequency)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Interactive: See how nearby vs distant timesteps are represented
# Compute cosine similarity between timestep embeddings

dim = 64
all_t = torch.arange(0, 1001, dtype=torch.float32)
all_emb = sinusoidal_embedding_1d(dim, all_t)

# Similarity matrix for selected timesteps
selected_t = [0, 100, 200, 500, 501, 502, 800, 999]
selected_emb = sinusoidal_embedding_1d(dim, torch.tensor(selected_t, dtype=torch.float32))
similarity = torch.nn.functional.cosine_similarity(
    selected_emb.unsqueeze(0), selected_emb.unsqueeze(1), dim=-1
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(similarity.numpy(), cmap='RdYlGn', vmin=-1, vmax=1, aspect='equal')
axes[0].set_xticks(range(len(selected_t)))
axes[0].set_xticklabels([f't={t}' for t in selected_t], rotation=45, fontsize=8)
axes[0].set_yticks(range(len(selected_t)))
axes[0].set_yticklabels([f't={t}' for t in selected_t], fontsize=8)
axes[0].set_title('Cosine Similarity Between Timestep Embeddings')
plt.colorbar(im, ax=axes[0])

ref_emb = sinusoidal_embedding_1d(dim, torch.tensor([500.0]))
sims = torch.nn.functional.cosine_similarity(all_emb, ref_emb, dim=-1)
axes[1].plot(all_t.numpy(), sims.numpy(), color='steelblue', linewidth=1.5)
axes[1].axvline(x=500, color='red', linestyle='--', alpha=0.5, label='t=500')
axes[1].set_xlabel('Timestep')
axes[1].set_ylabel('Cosine Similarity to t=500')
axes[1].set_title('Nearby timesteps have similar embeddings')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observations:")
print("  - t=500, t=501, t=502 have very high similarity (they're neighbors)")
print("  - t=0 and t=999 are very different from t=500")
print("  - The similarity drops smoothly with distance")

## 3.2 3D Rotary Position Embeddings (RoPE)

**The Problem:** After patchification, the model sees a flat sequence of tokens — it has no idea which token came from the top-left corner vs the bottom-right, or from frame 1 vs frame 5. We need to tell it **where** each token is.

**Why not just add position numbers?** (Absolute position embeddings)
- They don't generalize to different video sizes
- They can't express that "token 5 is 3 positions away from token 2" directly
- They waste capacity storing absolute locations rather than relative relationships

### RoPE Step by Step

RoPE encodes position by **rotating** pairs of dimensions. Let's build the intuition from scratch.

**Step 1: Think of pairs of dimensions as 2D points on a circle**

Take two consecutive dimensions from a Q or K vector, say (dim₀, dim₁). These form a 2D point. RoPE rotates this point by an angle that depends on the token's position:

```
Position 0: rotate by 0°
Position 1: rotate by θ°  
Position 2: rotate by 2θ°
Position 3: rotate by 3θ°
```

**Step 2: Different pairs rotate at different speeds**

Each pair of dimensions uses a different base angle θ. Some pairs rotate slowly (low frequency), others rotate fast (high frequency). This is exactly like the sinusoidal embeddings — but applied as rotations instead of additions.

**Step 3: The magic — dot product only sees relative rotation**

When computing attention, we take Q·K (dot product). If Q was rotated by angle 5θ (position 5) and K was rotated by angle 2θ (position 2), the dot product only depends on the **difference**: (5-2)θ = 3θ. The absolute positions cancel out!

This is the trigonometric identity: cos(α)cos(β) + sin(α)sin(β) = cos(α - β)

**Step 4: Extend to 3D for video**

Video tokens live in a 3D grid (Time, Height, Width). We split the head dimension into three parts and apply separate rotations for each axis:

```
Head dim (32) = Temporal (12) + Height (10) + Width (10)
```

Each axis gets its own set of rotation frequencies, so the model can independently reason about "same frame?" and "neighboring patch?"

In [ ]:
# Step 1: Visualize rotation on a circle
# RoPE rotates a 2D point (pair of dimensions) by a position-dependent angle

import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1: Show rotation for one frequency ---
ax = axes[0]
theta_base = 0.3  # base rotation angle (radians)
positions = range(8)

# Draw unit circle
circle = plt.Circle((0, 0), 1, fill=False, color='lightgray', linewidth=1)
ax.add_patch(circle)

# Original point (before rotation)
orig_x, orig_y = 0.8, 0.6  # some Q vector's pair of dims

colors = plt.cm.viridis(np.linspace(0, 1, len(positions)))
for pos in positions:
    angle = pos * theta_base
    # Rotation matrix: [cos -sin; sin cos]
    rot_x = orig_x * np.cos(angle) - orig_y * np.sin(angle)
    rot_y = orig_x * np.sin(angle) + orig_y * np.cos(angle)
    ax.scatter(rot_x, rot_y, color=colors[pos], s=80, zorder=5)
    ax.annotate(f'pos={pos}', (rot_x, rot_y), fontsize=7,
                textcoords="offset points", xytext=(5, 5))

ax.scatter(orig_x, orig_y, color='red', s=150, marker='*', zorder=6, label='Original (pos=0)')
ax.set_xlim(-1.4, 1.4)
ax.set_ylim(-1.4, 1.4)
ax.set_aspect('equal')
ax.axhline(y=0, color='gray', linewidth=0.5)
ax.axvline(x=0, color='gray', linewidth=0.5)
ax.set_title('Step 1: RoPE Rotates a 2D Point\n(one pair of dimensions, one frequency)', fontsize=11)
ax.set_xlabel('Dimension 0')
ax.set_ylabel('Dimension 1')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# --- Panel 2: Different frequencies rotate at different speeds ---
ax = axes[1]
freqs = [0.1, 0.3, 0.8]  # slow, medium, fast
freq_labels = ['Slow (low freq)', 'Medium', 'Fast (high freq)']
freq_colors = ['blue', 'orange', 'red']

for freq, label, color in zip(freqs, freq_labels, freq_colors):
    angles = [pos * freq for pos in range(16)]
    # Track the x-coordinate after rotation (shows oscillation)
    rot_xs = [orig_x * np.cos(a) - orig_y * np.sin(a) for a in angles]
    ax.plot(range(16), rot_xs, '-o', color=color, label=label, markersize=4, linewidth=2)

ax.set_xlabel('Token Position')
ax.set_ylabel('Dim 0 value after rotation')
ax.set_title('Step 2: Different Frequencies\n(each dim pair rotates at its own speed)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Panel 3: Show the actual rotation angles for 3D RoPE ---
ax = axes[2]
head_dim = 32

# Compute the actual angles (not complex, just the raw angles)
d_t = head_dim - 2 * (head_dim // 3)  # 12 dims -> 6 freq pairs
d_h = head_dim // 3   # 10 dims -> 5 freq pairs
d_w = head_dim // 3   # 10 dims -> 5 freq pairs

theta = 10000.0
t_freqs = 1.0 / (theta ** (torch.arange(0, d_t, 2)[:d_t//2].float() / d_t))
h_freqs = 1.0 / (theta ** (torch.arange(0, d_h, 2)[:d_h//2].float() / d_h))
w_freqs = 1.0 / (theta ** (torch.arange(0, d_w, 2)[:d_w//2].float() / d_w))

# Show rotation angle vs position for each axis
max_pos = 16
positions_range = torch.arange(max_pos).float()

for i, freq in enumerate(t_freqs[:3]):
    angles = (positions_range * freq).numpy()
    ax.plot(positions_range.numpy(), np.degrees(angles), '--',
            color=f'C{i}', linewidth=1.5, label=f'Temporal pair {i} (ω={freq:.4f})')

for i, freq in enumerate(h_freqs[:2]):
    angles = (positions_range * freq).numpy()
    ax.plot(positions_range.numpy(), np.degrees(angles), '-',
            color=f'C{i+3}', linewidth=1.5, label=f'Height pair {i} (ω={freq:.4f})')

ax.set_xlabel('Position along axis')
ax.set_ylabel('Rotation angle (degrees)')
ax.set_title('Step 4: Actual 3D RoPE Angles\n(each axis has its own frequencies)', fontsize=11)
ax.legend(fontsize=7, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Left: Same 2D point rotated to different positions — each position has a unique angle")
print("Middle: Slow frequencies change gradually, fast ones oscillate rapidly")
print("Right: The actual rotation angles used in our 3D RoPE (temporal + spatial)")

In [ ]:
# Step 3: PROVE that RoPE makes dot products depend only on RELATIVE distance
#
# We'll take the same Q and K vectors, place them at different absolute positions,
# but keep the DISTANCE between them fixed. The dot product should be identical.

head_dim = 32
freqs_t, freqs_h, freqs_w = precompute_freqs_cis_3d(head_dim)

torch.manual_seed(42)
q_base = torch.randn(1, 1, head_dim)
k_base = torch.randn(1, 1, head_dim)

def compute_rope_dot(q, k, freq_q, freq_k, hd):
    """Apply RoPE to q and k, then compute dot product."""
    d_t = hd - 2 * (hd // 3)
    d_h = hd // 3
    ones_h = torch.ones(1, 1, d_h // 2, dtype=freq_q.dtype)
    ones_w = torch.ones(1, 1, d_h // 2, dtype=freq_q.dtype)
    fq = torch.cat([freq_q[:, :, :d_t//2], ones_h, ones_w], dim=-1)
    fk = torch.cat([freq_k[:, :, :d_t//2], ones_h, ones_w], dim=-1)
    q_rot = rope_apply(q, fq, 1)
    k_rot = rope_apply(k, fk, 1)
    return (q_rot * k_rot).sum().item()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# --- Panel 1: Same distance, different absolute positions ---
ax = axes[0]
distances = [1, 3, 5, 10, 20]

for start in [0, 5, 15, 30]:
    dots = []
    for d in distances:
        fq = freqs_t[start:start+1].unsqueeze(0)
        fk = freqs_t[start+d:start+d+1].unsqueeze(0)
        dot = compute_rope_dot(q_base.clone(), k_base.clone(), fq, fk, head_dim)
        dots.append(dot)
    ax.plot(distances, dots, '-o', label=f'Q at pos {start}', linewidth=2, markersize=6)

ax.set_xlabel('Distance between Q and K', fontsize=12)
ax.set_ylabel('Q·K dot product (with RoPE)', fontsize=12)
ax.set_title('Same Distance → Same Dot Product\n(lines overlap = relative position only!)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# --- Panel 2: Contrast with what absolute encoding would look like ---
ax = axes[1]

# Without RoPE: absolute position embedding (add a position vector)
torch.manual_seed(42)
pos_embed = torch.randn(50, head_dim) * 0.1  # absolute position embeddings

for start in [0, 5, 15, 30]:
    dots = []
    for d in distances:
        q_abs = q_base + pos_embed[start:start+1].unsqueeze(0)
        k_abs = k_base + pos_embed[start+d:start+d+1].unsqueeze(0)
        dot = (q_abs * k_abs).sum().item()
        dots.append(dot)
    ax.plot(distances, dots, '-o', label=f'Q at pos {start}', linewidth=2, markersize=6)

ax.set_xlabel('Distance between Q and K', fontsize=12)
ax.set_ylabel('Q·K dot product (absolute pos)', fontsize=12)
ax.set_title('Absolute Position: Lines DON\'T Overlap\n(dot product depends on absolute positions)', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Left (RoPE):  All 4 lines overlap perfectly — only relative distance matters")
print("Right (Absolute): Lines are different — the dot product changes with absolute position")
print()
print("This is why RoPE is superior: the model automatically learns RELATIVE relationships")
print("without needing to figure out 'subtract position A from position B' on its own.")

### How RoPE Injection Actually Works — The Mechanics

We showed *why* RoPE works (relative position in dot products). Now let's see **exactly how** the rotation is applied to Q and K vectors.

#### The Position Problem

After patchify + flatten, tokens **lose their position** — the vector itself has no idea where it came from:

```
[B, dim, f, h, w]  →  reshape  →  [B, f*h*w, dim]
                                    token 0: was at (0,0,0)  ← but the vector doesn't know!
                                    token 1: was at (0,0,1)
                                    token 2: was at (0,0,2)
                                    ...
```

The Q and K projections (`Wq`, `Wk`) are just learned linear transforms — they also don't add any position info. So at this point, the model is completely blind to where tokens came from.

**RoPE stamps position back in from the outside**, right before the attention dot product. It works because **we track the mapping from token index to grid position** (the flatten order `f → h → w` is deterministic).

```
Patchify → tokens lose position (just a flat sequence)
     ↓
Q, K projection (Wq, Wk) → still no position info
     ↓
RoPE → we KNOW which token was where (deterministic ordering)
       rotate each Q, K by the RIGHT angle for its (t, h, w)
     ↓
Q·Kᵀ → relative position is now encoded in the dot product
```

#### The Rotation Angle: Position × Frequency

The rotation angle for each dimension pair has **two independent parts**:

| Part | What it is | Where it comes from | Changes with... |
|------|-----------|--------------------|-----------------|
| **Position** (e.g. 3) | The token's coordinate along one axis | The 3D grid mapping | Different tokens, different videos |
| **Frequency** (θᵢ) | A fixed constant per dimension pair | Formula: `θᵢ = 1 / 10000^(2i/d)` | Only the dimension pair index |

**angle = position × θᵢ**

Concrete example — a token at grid position `(t=2, h=3, w=1)` with `head_dim=32`:

```
Temporal pairs (dims 0-11):   angle = 2 × θᵢ    ← position along time axis
Height pairs  (dims 12-21):   angle = 3 × θᵢ    ← position along height axis
Width pairs   (dims 22-31):   angle = 1 × θᵢ    ← position along width axis
```

The grid tells us **which position number** to use (2, 3, or 1 depending on axis). The formula tells us **which θ** to use (depends on dimension pair index). Multiply them → rotation angle.

#### The Rotation Itself

RoPE pairs up consecutive elements of the head dimension: `(dim₀, dim₁)`, `(dim₂, dim₃)`, etc.

Each pair is treated as a **2D point** and rotated using complex multiplication:

```
(q₀, q₁) × (cos(m·θ) + i·sin(m·θ))

=  (q₀·cos(m·θ) - q₁·sin(m·θ),  q₀·sin(m·θ) + q₁·cos(m·θ))
    \_________________________/    \________________________________/
         new q₀                          new q₁
```

This is NOT a simple element-wise scaling — each output depends on **BOTH** input elements of the pair. That cross-mixing is what makes it a **rotation** (preserves vector magnitude, changes direction).

Let's verify all of this with actual numbers:

In [ ]:
# ============================================================
# NUMERICAL WALKTHROUGH: See exactly what RoPE does to Q values
# ============================================================
import numpy as np

# --- A single pair of Q dimensions at position m=10 ---
q0, q1 = 0.5, 0.3          # one pair from head_dim
theta = 0.01               # one specific frequency
m = 10                     # token position

angle = m * theta          # rotation angle = 0.1 radians

# The rotation formula:
new_q0 = q0 * np.cos(angle) - q1 * np.sin(angle)
new_q1 = q0 * np.sin(angle) + q1 * np.cos(angle)

print("=" * 60)
print("STEP-BY-STEP: RoPE rotation of one dimension pair")
print("=" * 60)
print(f"\nOriginal pair:  q = ({q0}, {q1})")
print(f"Position m = {m}, frequency theta = {theta}")
print(f"Rotation angle = m * theta = {m} * {theta} = {angle} radians")
print(f"\ncos({angle}) = {np.cos(angle):.6f}")
print(f"sin({angle}) = {np.sin(angle):.6f}")
print(f"\nnew_q0 = q0*cos - q1*sin = {q0}*{np.cos(angle):.6f} - {q1}*{np.sin(angle):.6f} = {new_q0:.6f}")
print(f"new_q1 = q0*sin + q1*cos = {q0}*{np.sin(angle):.6f} + {q1}*{np.cos(angle):.6f} = {new_q1:.6f}")
print(f"\nRotated pair:   q' = ({new_q0:.6f}, {new_q1:.6f})")
print(f"\nMagnitude before: {np.sqrt(q0**2 + q1**2):.6f}")
print(f"Magnitude after:  {np.sqrt(new_q0**2 + new_q1**2):.6f}")
print("Magnitude is PRESERVED (it's a rotation, not scaling!)")

# --- Now verify with actual PyTorch RoPE code ---
print("\n" + "=" * 60)
print("VERIFY: Using the actual rope_apply function")
print("=" * 60)

head_dim = 8  # small for clarity
q_vec = torch.tensor([[[0.5, 0.3, 0.7, 0.2, 0.1, 0.9, 0.4, 0.6]]])  # [1, 1, 8]
position = 10

# Build freq for a single position using 1D RoPE
from nano_video_gen.model.components import precompute_freqs_cis
freqs_1d = precompute_freqs_cis(head_dim, end=20)  # [20, 4] complex
freq_at_pos = freqs_1d[position:position+1].unsqueeze(0)  # [1, 1, 4] complex

q_rotated = rope_apply(q_vec, freq_at_pos, num_heads=1)

print(f"\nInput  Q: {q_vec.squeeze().tolist()}")
print(f"Output Q: [{', '.join(f'{v:.4f}' for v in q_rotated.squeeze().tolist())}]")
print(f"\nPair-by-pair breakdown (head_dim={head_dim}, {head_dim//2} pairs):")

thetas = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2)[:head_dim//2].float() / head_dim))
for i in range(head_dim // 2):
    idx0, idx1 = 2*i, 2*i+1
    a = position * thetas[i].item()
    orig = (q_vec[0, 0, idx0].item(), q_vec[0, 0, idx1].item())
    rotated = (q_rotated[0, 0, idx0].item(), q_rotated[0, 0, idx1].item())
    print(f"  Pair {i} (dims {idx0},{idx1}): theta={thetas[i]:.6f}, "
          f"angle={a:.4f} rad ({np.degrees(a):.1f} deg)")
    print(f"    ({orig[0]:.4f}, {orig[1]:.4f}) -> ({rotated[0]:.4f}, {rotated[1]:.4f})")

print("\nNotice: Pair 0 (high freq) rotates 573 deg = massive change")
print("        Pair 3 (low freq) rotates 0.6 deg = barely moves")
print("This multi-scale rotation is key to capturing both fine and coarse position.")

# --- 3D Grid Mapping: token index -> (t, h, w) -> rotation angles ---
print("\n" + "=" * 60)
print("3D GRID MAPPING: How token index becomes rotation angles")
print("=" * 60)

f, h, w = 4, 8, 8  # grid after patchify
print(f"\nVideo grid after patchify: f={f}, h={h}, w={w}")
print(f"Total tokens: {f*h*w}")
print(f"\nFlatten order is f -> h -> w, so:")

# Show first few token mappings
for token_idx in [0, 1, w-1, w, w*h, w*h+w+3]:
    t_pos = token_idx // (h * w)
    h_pos = (token_idx % (h * w)) // w
    w_pos = token_idx % w
    print(f"  Token {token_idx:3d} -> grid ({t_pos}, {h_pos}, {w_pos})")
    print(f"             temporal dims rotated by {t_pos} * theta")
    print(f"             height dims rotated by  {h_pos} * theta")
    print(f"             width dims rotated by   {w_pos} * theta")


In [ ]:
# ============================================================
# VISUALIZE: What RoPE does to each dimension pair
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1: Rotation of one pair across positions ---
ax = axes[0]
q0, q1 = 0.8, 0.3
theta_val = 0.3
positions_to_show = list(range(0, 22, 1))
colors = plt.cm.coolwarm(np.linspace(0, 1, len(positions_to_show)))

# Unit circle
circ = plt.Circle((0, 0), np.sqrt(q0**2 + q1**2), fill=False, color='lightgray', ls='--')
ax.add_patch(circ)

for idx, pos in enumerate(positions_to_show):
    angle = pos * theta_val
    rq0 = q0 * np.cos(angle) - q1 * np.sin(angle)
    rq1 = q0 * np.sin(angle) + q1 * np.cos(angle)
    ax.scatter(rq0, rq1, c=[colors[idx]], s=60, zorder=5, edgecolors='k', linewidth=0.5)
    if pos % 5 == 0:
        ax.annotate(f'm={pos}', (rq0, rq1), fontsize=7, textcoords='offset points', xytext=(5,5))

ax.scatter(q0, q1, c='red', s=150, marker='*', zorder=6, label='Original (m=0)')
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)
ax.set_xlabel('dim 0 value', fontsize=11)
ax.set_ylabel('dim 1 value', fontsize=11)
ax.set_title('One Pair Rotated Across Positions\n(stays on circle = magnitude preserved)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# --- Panel 2: Before vs after RoPE for a full head_dim vector ---
ax = axes[1]
head_dim = 32
torch.manual_seed(42)
q_full = torch.randn(1, 1, head_dim)

freqs_1d = precompute_freqs_cis(head_dim, end=50)
positions_demo = [0, 5, 15, 30]
colors_demo = ['#1565c0', '#e65100', '#2e7d32', '#c62828']

dims = np.arange(head_dim)
ax.bar(dims - 0.2, q_full.squeeze().numpy(), width=0.35, color='lightgray',
       edgecolor='gray', label='Original Q', alpha=0.7)

for pos, color in zip(positions_demo, colors_demo):
    freq = freqs_1d[pos:pos+1].unsqueeze(0)
    q_rot = rope_apply(q_full.clone(), freq, num_heads=1)
    ax.plot(dims, q_rot.squeeze().detach().numpy(), 'o-', color=color,
            markersize=3, lw=1.2, alpha=0.8, label=f'After RoPE (m={pos})')

ax.set_xlabel('Dimension index within head', fontsize=11)
ax.set_ylabel('Value', fontsize=11)
ax.set_title('Same Q Vector Rotated at Different Positions\n(each position gives a unique pattern)', fontsize=11)
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.2)

# --- Panel 3: 3D RoPE — which dims get which axis ---
ax = axes[2]
hd = 32
d_t = hd - 2 * (hd // 3)  # 12 dims = 6 pairs
d_h = hd // 3              # 10 dims = 5 pairs
d_w = hd // 3              # 10 dims = 5 pairs

# Color-code each dimension by its axis
colors_bar = []
labels_bar = []
for i in range(hd):
    if i < d_t:
        colors_bar.append('#e53935')  # red = temporal
        labels_bar.append('T')
    elif i < d_t + d_h:
        colors_bar.append('#43a047')  # green = height
        labels_bar.append('H')
    else:
        colors_bar.append('#1e88e5')  # blue = width
        labels_bar.append('W')

ax.bar(range(hd), [1]*hd, color=colors_bar, edgecolor='white', linewidth=0.5)

# Add pair brackets
for i in range(0, hd, 2):
    ax.plot([i, i+1], [-0.08, -0.08], 'k-', lw=2, clip_on=False)
    ax.text(i + 0.5, -0.15, f'P{i//2}', ha='center', fontsize=6, color='gray')

import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='#e53935', label=f'Temporal ({d_t} dims, {d_t//2} pairs)'),
    mpatches.Patch(color='#43a047', label=f'Height ({d_h} dims, {d_h//2} pairs)'),
    mpatches.Patch(color='#1e88e5', label=f'Width ({d_w} dims, {d_w//2} pairs)'),
], fontsize=9, loc='upper right')

ax.set_xlabel('Dimension index within head_dim', fontsize=11)
ax.set_title(f'3D RoPE: Head Dim Split by Axis\n(head_dim={hd}: {d_t}T + {d_h}H + {d_w}W)', fontsize=11)
ax.set_yticks([])
ax.set_xlim(-0.5, hd - 0.5)

plt.tight_layout()
plt.show()

print("Left:   Same (q0, q1) pair rotated at each position — traces a circle (magnitude preserved)")
print("Middle: Full head_dim vector after RoPE — each position produces a unique pattern")
print("Right:  3D RoPE splits head_dim into temporal/height/width groups, each rotated by its axis")

## 3.3 RMSNorm vs LayerNorm

**The Problem:** Deep neural networks suffer from **internal covariate shift** — as data flows through many layers, the distribution of activations can drift, making training unstable. Normalization fixes this by keeping activations in a well-behaved range.

**LayerNorm** (used in most Transformers):
$$\text{LayerNorm}(x) = \frac{x - \text{mean}(x)}{\text{std}(x)} \cdot \gamma + \beta$$
It centers (subtracts mean) AND normalizes (divides by std).

**RMSNorm** (used in Wan/LLaMA/Gemma):
$$\text{RMSNorm}(x) = \frac{x}{\text{RMS}(x)} \cdot \gamma \qquad \text{where } \text{RMS}(x) = \sqrt{\frac{1}{d}\sum_i x_i^2}$$
It ONLY normalizes by magnitude (no mean subtraction).

**Why does Wan use RMSNorm?**
1. **5-10% faster** — skips the mean computation (saves one reduction operation per layer)
2. **Works just as well** — research shows the centering step in LayerNorm is often unnecessary
3. **Proven at scale** — used by LLaMA, Gemma, and other state-of-the-art models
4. In Wan, RMSNorm is specifically applied to **Q and K** in attention (stabilizes the dot product magnitudes)

In [ ]:
dim = 128
rms_norm = RMSNorm(dim)
layer_norm = nn.LayerNorm(dim)

torch.manual_seed(42)
x = torch.randn(4, 16, dim) * 3.0 + 2.0

with torch.no_grad():
    x_rms = rms_norm(x)
    x_ln = layer_norm(x)

print(f"Input  -- mean: {x.mean():.4f}, std: {x.std():.4f}")
print(f"RMSNorm -- mean: {x_rms.mean():.4f}, std: {x_rms.std():.4f}")
print(f"LayerNorm -- mean: {x_ln.mean():.4f}, std: {x_ln.std():.4f}")
print()
print("RMSNorm does NOT center the data (mean != 0).")
print("LayerNorm centers (mean ~ 0) AND normalizes variance.")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].hist(x.flatten().numpy(), bins=80, alpha=0.7, color='gray', edgecolor='black')
axes[0].set_title(f'Input (mean={x.mean():.2f}, std={x.std():.2f})')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.5)

axes[1].hist(x_rms.flatten().numpy(), bins=80, alpha=0.7, color='steelblue', edgecolor='black')
axes[1].set_title(f'RMSNorm (mean={x_rms.mean():.2f}, std={x_rms.std():.2f})')
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)

axes[2].hist(x_ln.flatten().numpy(), bins=80, alpha=0.7, color='coral', edgecolor='black')
axes[2].set_title(f'LayerNorm (mean={x_ln.mean():.2f}, std={x_ln.std():.2f})')
axes[2].axvline(x=0, color='red', linestyle='--', alpha=0.5)

for ax in axes:
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.suptitle('RMSNorm vs LayerNorm: Output Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 3.4 AdaIN Modulation — The Timestep's "Volume Knob"

**The Problem:** The model needs to behave **differently at different noise levels**:
- At high noise ($t \approx 1000$): focus on coarse structure (overall shapes, layout)
- At low noise ($t \approx 0$): focus on fine details (textures, edges)

But all timesteps share the **same network weights**! How can one model behave differently?

**The Solution: AdaIN (Adaptive Instance Normalization)**

$$\text{output} = x \cdot (1 + \text{scale}) + \text{shift}$$

The `shift` and `scale` are predicted from the timestep embedding. This gives the timestep a "volume knob" on each layer:
- **scale** controls *amplification* — which features to boost or suppress
- **shift** controls *bias* — where to center the activations
- **gate** (separate parameter) controls *how much this layer contributes* to the output

**Why is this better than just concatenating the timestep?**
- Concatenation only affects the input; modulation affects **every layer**
- Each of the 6 parameters per block (shift/scale/gate × self-attn + FFN) can be different
- The model effectively becomes a **different network** for each noise level

**Design detail:** The modulation starts near identity ($\text{scale} \approx 0$, $\text{shift} \approx 0$) so initially each block does nothing special — it learns to modulate during training.

In [ ]:
dim = 128
torch.manual_seed(42)
x = torch.randn(1, 16, dim)
shift = torch.randn(1, 1, dim) * 0.5
scale = torch.randn(1, 1, dim) * 0.3

x_modulated = modulate(x, shift, scale)

print(f"Input  -- mean: {x.mean():.4f}, std: {x.std():.4f}")
print(f"Output -- mean: {x_modulated.mean():.4f}, std: {x_modulated.std():.4f}")

fig = visualize_modulation_effect(x, x_modulated, shift, scale)
plt.show()

In [ ]:
# Interactive: See how modulation parameters change per dimension

dim = 128
torch.manual_seed(42)
x = torch.randn(1, 16, dim)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

scenarios = [
    ('High noise (t\u22481000)\nBig, bold adjustments', 1.0, 0.8),
    ('Medium noise (t\u2248500)\nModerate adjustments', 0.5, 0.3),
    ('Low noise (t\u2248100)\nSubtle refinements', 0.1, 0.05),
]

for col, (label, shift_mag, scale_mag) in enumerate(scenarios):
    torch.manual_seed(42)
    shift = torch.randn(1, 1, dim) * shift_mag
    scale = torch.randn(1, 1, dim) * scale_mag
    x_mod = modulate(x, shift, scale)

    axes[0, col].bar(range(32), x[0, 0, :32].numpy(), alpha=0.5, label='Before', color='blue', width=0.4)
    axes[0, col].bar([i+0.4 for i in range(32)], x_mod[0, 0, :32].detach().numpy(),
                     alpha=0.5, label='After', color='red', width=0.4)
    axes[0, col].set_title(label, fontsize=10)
    axes[0, col].legend(fontsize=8)
    axes[0, col].set_xlabel('Dimension (first 32)')

    scale_factors = (1 + scale[0, 0, :32]).numpy()
    colors_list = ['green' if s > 1 else 'red' for s in scale_factors]
    axes[1, col].bar(range(32), scale_factors, color=colors_list, alpha=0.7)
    axes[1, col].axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='No change')
    axes[1, col].set_title('Scale factors (1+scale)', fontsize=10)
    axes[1, col].set_xlabel('Dimension (first 32)')
    axes[1, col].legend(fontsize=8)

plt.suptitle('AdaIN Modulation at Different Noise Levels', fontsize=14)
plt.tight_layout()
plt.show()

print('At high noise: the model makes bold adjustments (large shifts and scales)')
print('At low noise: the model makes subtle refinements (small shifts and scales)')
print('Green bars = amplified features, Red bars = suppressed features')

## 3.5 Self-Attention — Every Patch Talks to Every Patch

**The Problem:** A video patch showing a person's hand needs to "know" where the person's face is (to maintain consistency), what happened in the previous frame (to ensure smooth motion), and what the background looks like (to avoid floating objects). How can each local patch get this global information?

**The Solution: Self-Attention**

Each patch creates three vectors:
- **Query (Q):** "What am I looking for?"
- **Key (K):** "What do I contain?"
- **Value (V):** "What information can I share?"

The attention mechanism:
1. Each token's Q is compared against every token's K (dot product)
2. The scores are softmax'd into weights (who to pay attention to)
3. Each token receives a weighted sum of all V vectors

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

**Key design choices in Wan:**
- **Bidirectional** (unlike GPT which is causal) — every patch can see every other patch
- **Q/K are RMSNorm'd** before attention — prevents dot products from exploding in magnitude
- **3D RoPE applied** to Q and K — encodes spatial-temporal position
- **Multi-head** — 4 heads in Nano, 40 in Wan 14B, each learning different attention patterns

**Why is this critical for video?** Unlike images, video needs **temporal consistency**. Self-attention across frames ensures that frame 5 is consistent with frame 1 — the same objects appear in the same places, just moved slightly.

In [ ]:
dim = 128
num_heads = 4
head_dim = dim // num_heads

self_attn = SelfAttention(dim=dim, num_heads=num_heads)
self_attn.eval()

torch.manual_seed(42)
f, h, w = 2, 4, 4
seq_len = f * h * w  # 32 tokens
x = torch.randn(1, seq_len, dim)

# Build 3D RoPE frequencies
freqs_t, freqs_h, freqs_w = precompute_freqs_cis_3d(head_dim)
freqs = torch.cat([
    freqs_t[:f].view(f, 1, 1, -1).expand(f, h, w, -1),
    freqs_h[:h].view(1, h, 1, -1).expand(f, h, w, -1),
    freqs_w[:w].view(1, 1, w, -1).expand(f, h, w, -1)
], dim=-1).reshape(seq_len, 1, -1)

with torch.no_grad():
    output = self_attn(x, freqs)
    # Compute attention weights for visualization
    q = self_attn.norm_q(self_attn.q(x))
    k = self_attn.norm_k(self_attn.k(x))
    q = rope_apply(q, freqs, num_heads)
    k = rope_apply(k, freqs, num_heads)
    q = rearrange(q, "b s (n d) -> b n s d", n=num_heads)
    k = rearrange(k, "b s (n d) -> b n s d", n=num_heads)
    attn_weights = F.softmax(torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5), dim=-1)

print(f"Input: {list(x.shape)} -> Output: {list(output.shape)}")
print(f"Attention weights: {list(attn_weights.shape)}")

fig = visualize_attention_maps(
    attn_weights,
    title=f"Self-Attention Maps ({seq_len} tokens = {f}f x {h}h x {w}w)",
    num_heads_to_show=4,
)
plt.show()

In [ ]:
# Interactive: Visualize what self-attention looks like in a 3D video grid
# Our 32 tokens = 2 frames x 4 height x 4 width

positions = []
for f_idx in range(f):
    for h_idx in range(h):
        for w_idx in range(w):
            positions.append(f'f{f_idx}h{h_idx}w{w_idx}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

query_tokens = [0, 8, 16]
query_labels = [
    f'Token 0 ({positions[0]})\nFrame 0, top-left',
    f'Token 8 ({positions[8]})\nFrame 0, middle',
    f'Token 16 ({positions[16]})\nFrame 1, top-left',
]

for ax, q_idx, label in zip(axes, query_tokens, query_labels):
    weights = attn_weights[0, 0, q_idx].numpy()
    weights_2d = weights.reshape(f, h * w)

    im = ax.imshow(weights_2d, cmap='hot', aspect='auto')
    ax.set_xlabel('Spatial position (within frame)')
    ax.set_ylabel('Frame')
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Frame 0', 'Frame 1'])
    ax.set_title(label, fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Self-Attention: Where Does Each Token Look? (Head 0)', fontsize=14)
plt.tight_layout()
plt.show()

print('Each row shows how much a query token attends to tokens in each frame.')
print('Brighter = higher attention weight.')
print('Notice how tokens may attend both within their frame AND across frames.')

## 3.6 Cross-Attention — How Text Guides Video Generation

**The Problem:** We've seen how self-attention lets video patches communicate with each other. But how does the model know *what* to generate? The text prompt "a cat playing with a ball" needs to somehow influence what appears in the video.

**The Solution: Cross-Attention**

Cross-attention is like self-attention, but the **Q comes from video** and **K, V come from text**:
- **Q** (from video): "What kind of information do I need at this position?"
- **K** (from text): "What concepts does the text describe?"
- **V** (from text): "What semantic features should be injected?"

```
Video tokens (32)  →  Q  ─┐
                           ├─→  Attention  →  Updated video tokens
Text tokens (8)   →  K,V ─┘
```

**Key differences from self-attention:**
- No RoPE (text tokens have their own positional encoding from T5)
- Asymmetric — video tokens attend to text, not the other way around
- Each video patch independently "reads" from the text, pulling in relevant semantics

**Why this matters:** A patch in the upper-right corner might attend strongly to the word "sky" in the prompt, while a patch at the bottom attends to "grass". This is how the model learns spatial-semantic associations.

In [ ]:
cross_attn = CrossAttention(dim=dim, num_heads=num_heads)
cross_attn.eval()

torch.manual_seed(42)
x_video = torch.randn(1, seq_len, dim)
seq_len_text = 8
context = torch.randn(1, seq_len_text, dim)

with torch.no_grad():
    output = cross_attn(x_video, context)
    q = cross_attn.norm_q(cross_attn.q(x_video))
    k = cross_attn.norm_k(cross_attn.k(context))
    q = rearrange(q, 'b s (n d) -> b n s d', n=num_heads)
    k = rearrange(k, 'b s (n d) -> b n s d', n=num_heads)
    cross_attn_weights = F.softmax(torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5), dim=-1)

print(f'Video input: {list(x_video.shape)} -> Output: {list(output.shape)}')
print(f'Cross-attention: {seq_len} video tokens attend to {seq_len_text} text tokens')

fig = visualize_attention_maps(
    cross_attn_weights,
    title=f'Cross-Attention Maps ({seq_len} video -> {seq_len_text} text tokens)',
    num_heads_to_show=4,
)
plt.show()

## 3.7 Feed-Forward Network (FFN) — Per-Token Processing Power

**The Problem:** Attention is great at *mixing information between tokens*, but it performs no per-token computation. A token that has gathered context from its neighbors still needs to *process* that combined information to produce useful features.

**The Solution: FFN (Feed-Forward Network)**

The FFN is a simple 2-layer MLP applied to each token independently:
$$\text{FFN}(x) = \text{Linear}_2(\text{GELU}(\text{Linear}_1(x)))$$

- **Linear₁** (dim → ffn_dim): Expands to a wider space (128 → 512, a 4x expansion)
- **GELU**: Non-linear activation (the "thinking" step)
- **Linear₂** (ffn_dim → dim): Projects back to model dimension

**Why GELU and not ReLU?**
- **ReLU**: $\max(0, x)$ — hard cutoff at zero, kills negative values completely
- **GELU**: Gaussian Error Linear Unit — *smooth* cutoff, small negative values can still pass through

GELU is preferred in modern Transformers because the smooth curve leads to better gradient flow during training. It's used by GPT, BERT, Wan, and most state-of-the-art models.

In [ ]:
# Interactive: Compare GELU vs ReLU activation functions
x_range = torch.linspace(-4, 4, 200)
relu_vals = torch.nn.functional.relu(x_range)
gelu_vals = torch.nn.functional.gelu(x_range, approximate='tanh')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x_range.numpy(), relu_vals.numpy(), label='ReLU', linewidth=2, color='red')
axes[0].plot(x_range.numpy(), gelu_vals.numpy(), label='GELU', linewidth=2, color='blue')
axes[0].plot(x_range.numpy(), x_range.numpy(), '--', alpha=0.3, color='gray', label='Identity')
axes[0].axhline(y=0, color='black', linewidth=0.5)
axes[0].axvline(x=0, color='black', linewidth=0.5)
axes[0].set_title('Activation Functions')
axes[0].set_xlabel('Input')
axes[0].set_ylabel('Output')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

x_zoom = torch.linspace(-2, 1, 200)
relu_zoom = torch.nn.functional.relu(x_zoom)
gelu_zoom = torch.nn.functional.gelu(x_zoom, approximate='tanh')
axes[1].plot(x_zoom.numpy(), relu_zoom.numpy(), label='ReLU', linewidth=2, color='red')
axes[1].plot(x_zoom.numpy(), gelu_zoom.numpy(), label='GELU', linewidth=2, color='blue')
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].axvline(x=0, color='black', linewidth=0.5)
axes[1].fill_between(x_zoom.numpy(), relu_zoom.numpy(), gelu_zoom.numpy(),
                     alpha=0.2, color='green', label='Difference')
axes[1].set_title('Zoomed: GELU allows small negative values through')
axes[1].set_xlabel('Input')
axes[1].set_ylabel('Output')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Why GELU? Smooth gradients lead to better training', fontsize=14)
plt.tight_layout()
plt.show()

print('ReLU: hard kink at x=0, gradient=0 for all negative inputs (dead neurons)')
print('GELU: smooth transition, gradients flow even for slightly negative values')

In [ ]:
ffn_dim = 512
ffn = nn.Sequential(
    nn.Linear(dim, ffn_dim),
    nn.GELU(approximate='tanh'),
    nn.Linear(ffn_dim, dim)
)

torch.manual_seed(42)
x = torch.randn(1, seq_len, dim)

with torch.no_grad():
    h = ffn[0](x)
    h_gelu = ffn[1](h)
    output = ffn[2](h_gelu)

print(f"Input: {list(x.shape)} -> After Linear1: {list(h.shape)} -> After GELU: {list(h_gelu.shape)} -> Output: {list(output.shape)}")

fig = visualize_ffn_activations(h_gelu, title="FFN Hidden Activations (after GELU)")
plt.show()

## 3.8 Gate Module — Learned Residual Connections

**The Problem:** In a standard residual connection ($\text{output} = x + f(x)$), every layer contributes equally. But in a diffusion model, some layers might be more important at certain noise levels:
- At high noise, maybe layer 1 (which does coarse structure) should contribute a lot
- At low noise, maybe layer 1 should barely contribute (let other layers handle fine details)

**The Solution: Gating**

$$\text{output} = x + \text{gate} \cdot f(x)$$

The **gate** value is predicted from the timestep embedding, so it's different for each noise level:
- `gate ≈ 0`: This layer's output is ignored (the residual connection is "off")
- `gate ≈ 1`: Standard residual connection
- `gate > 1`: This layer's output is amplified
- `gate < 0`: This layer's output is subtracted (rare but possible)

**Why this matters:**
- Each of the 6 modulation parameters per block includes a gate for both self-attention and FFN
- The model learns during training which layers to "turn on" at which noise levels
- This is much more flexible than a fixed residual connection

In [ ]:
gate_module = GateModule()

torch.manual_seed(42)
x = torch.randn(1, seq_len, dim)
residual = torch.randn(1, seq_len, dim)

gate_values = [0.0, 0.5, 1.0, 2.0, -0.5]
fig, axes = plt.subplots(1, len(gate_values), figsize=(16, 3))

for i, g in enumerate(gate_values):
    gate = torch.full((1, 1, dim), g)
    output = gate_module(x, gate, residual)
    diff = (output - x).flatten().detach().numpy()
    axes[i].hist(diff, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
    axes[i].set_title(f'gate = {g}')
    axes[i].set_xlabel('output - x')
    axes[i].axvline(x=0, color='red', linestyle='--', alpha=0.5)
    axes[i].set_xlim(-6, 6)

axes[0].set_ylabel('Count')
plt.suptitle('GateModule: output = x + gate * residual', fontsize=14)
plt.tight_layout()
plt.show()

print("gate=0.0: No residual (output = x)")
print("gate=1.0: Standard residual (output = x + residual)")
print("gate=2.0: Amplified residual")
print("gate=-0.5: Subtractive gating")

### Building Blocks Summary — Why Each Component Was Chosen

| Component | What it does | Why this choice? |
|-----------|-------------|-----------------|
| **Sinusoidal Embedding** | Encodes scalar timestep → vector | Smooth, unique, infinite range — proven since original Transformer |
| **3D RoPE** | Encodes (t, h, w) position | Relative position awareness + independent axes + generalizes to new sizes |
| **RMSNorm** | Normalizes activations | 5-10% faster than LayerNorm, proven at scale (LLaMA, Gemma) |
| **AdaIN Modulation** | Timestep controls each layer | Per-layer, per-feature control — model becomes different net at each noise level |
| **Self-Attention** | Patches communicate globally | Video needs temporal consistency — patches across frames must agree |
| **Cross-Attention** | Text guides generation | Each video patch reads relevant text tokens independently |
| **FFN (GELU)** | Per-token processing | GELU has smooth gradients → better training than ReLU |
| **Gating** | Learned residual strength | Flexible — some layers matter more at certain noise levels |

Every component in Wan 2.1 exists for a reason. Our Nano model uses **exactly the same components** — just smaller. This means the architectural lessons transfer directly to understanding the full-scale model.

---
# Section 4: DiT Architecture

The **NanoDiT** assembles all building blocks into a complete Diffusion Transformer. Let's understand the key architectural choices that make it different from a standard LLM Transformer.

| Aspect | LLM Transformer | DiT (Diffusion Transformer) |
|--------|-----------------|-----|\
| Input | Token embeddings | **Patchified** noisy latents |
| Self-attention | Causal (masked) — can only see past | **Bidirectional** — sees everything |
| Conditioning | None / prefix | Timestep via **AdaIN modulation** |
| Cross-attention | None (in decoder-only) | **Text conditioning** |
| Positional encoding | 1D | **3D RoPE** (time + height + width) |
| Output | Next-token logits | Predicted **velocity** $v = \epsilon - x_0$ |

### Patchification: From Pixels to Tokens

The first step is converting the 3D latent volume into a sequence of tokens that the Transformer can process. This is done via a **3D convolution** with `kernel_size = stride = (1, 2, 2)`:

```
Latent:  [B, 16, 5, 16, 16]     (16 channels, 5 frames, 16×16 spatial)
         ↓ Conv3d(16→128, kernel=(1,2,2), stride=(1,2,2))
Patches: [B, 128, 5, 8, 8]      (128-dim embeddings, 5×8×8 = 320 patches)
         ↓ Reshape
Tokens:  [B, 320, 128]           (320 tokens, each 128-dimensional)
```

**Why patches instead of individual latent pixels?**
- Self-attention is $O(n^2)$ — with 5×16×16 = 1280 pixels, that's 1.6M attention pairs
- With 5×8×8 = 320 patches (2×2 spatial grouping), it's only 102K pairs — **16× fewer!**
- Each patch captures a small local region, and attention handles global relationships

In [ ]:
from nano_video_gen.model.nano_dit import NanoDiT

# NanoDiT with real dimensions: 16 latent channels from Wan VAE, 4096-dim T5 embeddings
demo_model = NanoDiT(
    dim=128, in_dim=16, ffn_dim=512, out_dim=16,
    text_dim=4096, freq_dim=64, num_heads=4, num_layers=2,
    patch_size=(1, 2, 2),
)
demo_model.eval()

print("=" * 70)
print("NanoDiT Model Architecture")
print("  in_dim=16 (Wan VAE latent channels)")
print("  text_dim=4096 (T5 umt5-xxl embeddings)")
print("=" * 70)
print(demo_model)

In [ ]:
# Interactive: Visualize how patchification works
# Show the 3D latent being divided into patches

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Show the latent grid (one frame, showing 16x16 spatial with patches highlighted)
ax = axes[0]
ax.set_xlim(0, 16)
ax.set_ylim(0, 16)
ax.set_aspect('equal')
ax.invert_yaxis()

# Draw 2x2 patches with alternating colors
colors = ['#FFE0B2', '#B3E5FC', '#C8E6C9', '#F8BBD0',
          '#D1C4E9', '#FFE0B2', '#B3E5FC', '#C8E6C9']
patch_idx = 0
for row in range(0, 16, 2):
    for col in range(0, 16, 2):
        color = colors[patch_idx % len(colors)]
        rect = plt.Rectangle((col, row), 2, 2, linewidth=1.5,
                             edgecolor='black', facecolor=color, alpha=0.7)
        ax.add_patch(rect)
        ax.text(col + 1, row + 1, str(patch_idx), ha='center', va='center', fontsize=6)
        patch_idx += 1

ax.set_title(f'Spatial Patchification (16×16 → 8×8)\nPatch size = 2×2, {patch_idx} patches per frame', fontsize=10)
ax.set_xlabel('Width')
ax.set_ylabel('Height')

# 2. Show temporal dimension
ax = axes[1]
frame_colors = ['#FFE0B2', '#B3E5FC', '#C8E6C9', '#F8BBD0', '#D1C4E9']
for f_idx in range(5):
    y = f_idx * 1.5
    rect = plt.Rectangle((0, y), 8, 1, linewidth=1.5,
                         edgecolor='black', facecolor=frame_colors[f_idx], alpha=0.7)
    ax.add_patch(rect)
    ax.text(4, y + 0.5, f'Frame {f_idx}: 8×8 = 64 patches', ha='center', va='center', fontsize=9)
ax.set_xlim(-0.5, 9)
ax.set_ylim(-0.5, 8)
ax.set_title(f'Temporal: 5 frames (no temporal patching)\nPatch size t=1, all frames kept', fontsize=10)
ax.axis('off')

# 3. Summary: total token count
ax = axes[2]
ax.axis('off')
summary = (
    "Patchification Summary\n"
    "━━━━━━━━━━━━━━━━━━━━━━━\n\n"
    "Input:  [B, 16, 5, 16, 16]\n"
    "  16 latent channels\n"
    "  5 temporal frames\n"
    "  16×16 spatial\n\n"
    "Patch size: (1, 2, 2)\n"
    "  t: 5/1 = 5 frames\n"
    "  h: 16/2 = 8 patches\n"
    "  w: 16/2 = 8 patches\n\n"
    "Total: 5 × 8 × 8 = 320 tokens\n"
    "Each token: 128-dimensional\n\n"
    "Output: [B, 320, 128]"
)
ax.text(0.1, 0.5, summary, transform=ax.transAxes, fontsize=11,
        verticalalignment='center', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.suptitle('How Patchification Converts 3D Video Latents to Transformer Tokens', fontsize=14)
plt.tight_layout()
plt.show()

### Anatomy of a Single DiT Block

Each DiT block processes video tokens through three stages, all conditioned on the timestep via AdaIN modulation. Here's the exact data flow:

```
Input x (B, 320, 128)
  │
  ├── LayerNorm (no affine) ──→ AdaIN(shift_msa, scale_msa) ──→ Self-Attention(+RoPE) ──→ × gate_msa
  │                                                                                          │
  └────────────────────────────────────────────────── + ◄────────────────────────────────────┘
  │
  ├── LayerNorm (with affine) ──→ Cross-Attention(text) ──→────────────────────────┐
  │                                                                                │
  └────────────────────────────────────────────────────── + ◄──────────────────────┘
  │
  ├── LayerNorm (no affine) ──→ AdaIN(shift_mlp, scale_mlp) ──→ FFN ──→ × gate_mlp
  │                                                                         │
  └──────────────────────────────────────────────────── + ◄────────────────┘
  │
Output x (B, 320, 128)
```

**Why three stages?**
1. **Self-attention** → tokens gather information from each other (global context)
2. **Cross-attention** → tokens read from text embeddings (semantic guidance)
3. **FFN** → each token processes its gathered information independently (feature transform)

**Why is cross-attention not modulated but self-attention and FFN are?**
This is a design choice from Wan — the text conditioning is always "on" regardless of noise level, while the internal processing adapts to the current denoising step.

In [ ]:
# Parameter breakdown
def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

components = {
    'patch_embedding': demo_model.patch_embedding,
    'text_embedding':  demo_model.text_embedding,
    'time_embedding':  demo_model.time_embedding,
    'time_projection': demo_model.time_projection,
}
for i, block in enumerate(demo_model.blocks):
    components[f'blocks[{i}]'] = block
components['head'] = demo_model.head

total = count_params(demo_model)
print(f"{'Component':<22} {'Parameters':>12}  {'% of Total':>10}")
print('-' * 48)
for name, mod in components.items():
    n = count_params(mod)
    print(f'{name:<22} {n:>12,}  {n / total * 100:>9.1f}%')
print('-' * 48)
print(f"{'TOTAL':<22} {total:>12,}  100.0%")
print()
print('Note: text_embedding is large because it projects 4096-dim T5')
print('embeddings down to 128-dim (Linear(4096, 128) + Linear(128, 128)).')

In [ ]:
# Forward pass with real dimensions
B = 1
x = torch.randn(B, 16, 5, 16, 16)        # [B, 16, T', H', W'] -- Wan VAE latent shape
timestep = torch.tensor([500.0])           # [B]
context = torch.randn(B, 226, 4096)        # [B, seq_len, 4096] -- T5 embedding shape

print(f"Input:    x={list(x.shape)}, timestep={timestep.item():.0f}, context={list(context.shape)}")
print(f"  x: 16 latent channels, 5 temporal frames, 16x16 spatial")
print(f"  context: 226 text tokens, 4096-dim T5 embeddings")

with torch.no_grad():
    output = demo_model(x, timestep, context)

print(f"Output:   {list(output.shape)}")
print(f"Same dims as input: {list(x.shape) == list(output.shape)}")
print(f"\nThe model predicts a velocity field v with the same shape as the input latent.")

In [ ]:
# Data flow visualization
from nano_video_gen.visualization.viz import visualize_data_flow

fig = visualize_data_flow(demo_model, x, timestep, context, figsize=(16, 5))
plt.show()

In [ ]:
import math

def estimate_params(
    dim, ffn_dim, num_layers, num_heads,
    in_dim, out_dim, text_dim, freq_dim,
    patch_size=(1, 2, 2),
):
    patch_vol = math.prod(patch_size)
    p_patch = in_dim * dim * patch_vol + dim
    p_text = (text_dim * dim + dim) + (dim * dim + dim)
    p_time = (freq_dim * dim + dim) + (dim * dim + dim)
    p_time_proj = dim * (dim * 6) + (dim * 6)
    p_self_attn = 4 * (dim * dim + dim) + 2 * dim
    p_cross_attn = 4 * (dim * dim + dim) + 2 * dim
    p_ffn = (dim * ffn_dim + ffn_dim) + (ffn_dim * dim + dim)
    p_norms = dim + dim
    p_mod = 6 * dim
    p_per_block = p_self_attn + p_cross_attn + p_ffn + p_norms + p_mod
    p_blocks = p_per_block * num_layers
    p_head = (dim * (out_dim * patch_vol) + out_dim * patch_vol) + 2 * dim
    total = p_patch + p_text + p_time + p_time_proj + p_blocks + p_head
    return {"TOTAL": total}

configs = {
    "Nano (~2.0M)": dict(dim=128, ffn_dim=512, num_layers=2, num_heads=4, in_dim=16, out_dim=16, text_dim=4096, freq_dim=64),
    "Wan 1.3B": dict(dim=1536, ffn_dim=8960, num_layers=30, num_heads=12, in_dim=16, out_dim=16, text_dim=4096, freq_dim=256),
    "Wan 14B": dict(dim=5120, ffn_dim=13824, num_layers=40, num_heads=40, in_dim=16, out_dim=16, text_dim=4096, freq_dim=256),
}

results = {name: estimate_params(**cfg) for name, cfg in configs.items()}
nano_total = results["Nano (~2.0M)"]["TOTAL"]

print("--- Scaling Comparison ---")
for name in configs:
    t = results[name]["TOTAL"]
    print(f"  {name:>14}: {t:>14,} params  ({t / nano_total:,.0f}x vs Nano)")

print(f"\nKey insight: dim 128 -> 5120 is {5120//128}x width. Since params scale as O(dim^2),")
print(f"this alone gives ~{(5120/128)**2:.0f}x more params. Combined with 40 layers (vs 2) = ~{results['Wan 14B']['TOTAL'] / nano_total:,.0f}x total.")

---
# Section 5: Flow Matching — The Diffusion Paradigm

**Flow matching** is the training paradigm used by Wan 2.1 (and FLUX, Stable Diffusion 3). It replaced the older DDPM approach used in earlier models.

### The Core Idea: Noise as Navigation

Imagine you're in a dense fog (pure noise). You want to reach a specific destination (your target video). Flow matching teaches the model to be a **compass** that always points toward the destination:

1. **Training:** Show the model many pairs of (foggy scene, clear destination). At each training step:
   - Start with a clean video $x_0$
   - Add a random amount of fog: $x_t = (1-t) \cdot x_0 + t \cdot \text{noise}$
   - Ask the model: "Which direction is the clean video?"
   - The correct answer is the **velocity**: $v = \text{noise} - x_0$ (the direction from clean to noisy)

2. **Inference:** Start from pure noise and follow the compass:
   - At each step, the model predicts "go this way" (the velocity)
   - Take a small step in that direction (Euler integration)
   - Repeat 20-50 times until you arrive at a clean video

### Why Flow Matching Over DDPM?

| Aspect | DDPM (older) | Flow Matching (Wan 2.1) |
|--------|------|---------------|
| Forward process | Complex: $x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$ | Simple: $x_t = (1-t) x_0 + t \epsilon$ |
| Training target | Predict noise $\epsilon$ | Predict velocity $v = \epsilon - x_0$ |
| Denoising step | Complex variance formula with many hyperparameters | Simple Euler step: $x_{t-1} = x_t + v \cdot \Delta\sigma$ |
| Steps needed | 1000 steps originally, 20-50 with tricks | 20-50 steps natively |
| Path | Curved path in latent space | **Straight** path (linear interpolation) → faster convergence |

### What Does the "Shift" Parameter Do?

The sigma schedule controls **how fast the noise decreases** at each step. Not all noise levels are equally important:
- **High noise** (early steps): The model is making big structural decisions — "Where should the sky be? Where's the ground?"
- **Low noise** (late steps): The model is adding fine details — "What texture does this surface have?"

The **shift** parameter biases the schedule to spend more compute on harder (high-noise) steps:
- `shift=1`: Equal time at all noise levels (linear)
- `shift=5` (Wan default): ~60% of steps at high noise — more time for structure

This is like a painter who spends most of their time on the sketch and composition, then quickly adds details at the end.

In [ ]:
from nano_video_gen.diffusion.flow_match import FlowMatchScheduler

# Compare sigma schedules for different shift values
shifts = [1, 3, 5]
num_steps = 50

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#2196F3', '#FF9800', '#E91E63']

for shift, color in zip(shifts, colors):
    scheduler = FlowMatchScheduler()
    scheduler.set_timesteps(num_inference_steps=num_steps, shift=shift)
    sigmas = scheduler.sigmas.numpy()
    timesteps_arr = scheduler.timesteps.numpy()
    step_indices = np.arange(len(sigmas))

    axes[0].plot(step_indices, sigmas, '-o', color=color, markersize=3, label=f'shift={shift}', linewidth=2)
    axes[1].plot(step_indices, timesteps_arr, '-o', color=color, markersize=3, label=f'shift={shift}', linewidth=2)

axes[0].set_title('Sigma (noise level) vs Step Index', fontsize=13)
axes[0].set_xlabel('Denoising step')
axes[0].set_ylabel('Sigma')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.02, 1.05)

axes[1].set_title('Timestep vs Step Index', fontsize=13)
axes[1].set_xlabel('Denoising step')
axes[1].set_ylabel('Timestep (0-1000)')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

fig.suptitle('Effect of Shift on the Noise Schedule', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("With shift=5 (Wan default), ~60% of steps are at sigma > 0.5,")
print("giving more compute for the hardest (high-noise) denoising steps.")

In [ ]:
# Visualize the forward noising process
H, W = 64, 64
y_grid, x_grid = torch.meshgrid(torch.linspace(-1, 1, H), torch.linspace(-1, 1, W), indexing='ij')
checker = ((torch.floor(x_grid * 4) + torch.floor(y_grid * 4)) % 2).float()
circle = 1.0 - (x_grid ** 2 + y_grid ** 2).sqrt().clamp(0, 1)
x0 = torch.stack([checker, circle, 0.5 * (checker + circle)], dim=0) * 2 - 1

torch.manual_seed(42)
noise = torch.randn_like(x0)

t_values = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, len(t_values), figsize=(16, 3.5))

for ax, t in zip(axes, t_values):
    x_t = (1 - t) * x0 + t * noise
    img = ((x_t + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(f'$t = {t:.2f}$', fontsize=12)
    ax.axis('off')

fig.suptitle('Forward Process: $x_t = (1-t) \\cdot x_0 + t \\cdot \\epsilon$', fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

### Understanding the Velocity Target

In flow matching, the model predicts a **velocity** $v = \epsilon - x_0$. This velocity points from the clean data toward the noise. During inference, we follow this velocity *in reverse* to go from noise back to clean data.

Let's visualize this in 2D to build intuition:

In [ ]:
# Visualize the flow matching process in 2D
# This makes the abstract math concrete and visual

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Example: one clean point and one noise point
np.random.seed(42)
x0 = np.array([2.0, 1.0])   # Clean data point
eps = np.array([-1.0, 2.0])   # Noise point
velocity = eps - x0            # True velocity

# 1. Show the linear interpolation path
ax = axes[0]
t_vals = np.linspace(0, 1, 20)
path_x = [(1 - t) * x0[0] + t * eps[0] for t in t_vals]
path_y = [(1 - t) * x0[1] + t * eps[1] for t in t_vals]
ax.plot(path_x, path_y, 'b-', linewidth=2, alpha=0.5)
ax.scatter(*x0, color='green', s=150, zorder=5, label='$x_0$ (clean)', marker='*')
ax.scatter(*eps, color='red', s=150, zorder=5, label='$\\epsilon$ (noise)', marker='X')

# Mark intermediate points
for t in [0.25, 0.5, 0.75]:
    xt = (1 - t) * x0 + t * eps
    ax.scatter(*xt, color='orange', s=60, zorder=4)
    ax.annotate(f't={t}', xt, fontsize=9, textcoords="offset points", xytext=(5, 5))

ax.set_title('Forward Process\n$x_t = (1-t) \\cdot x_0 + t \\cdot \\epsilon$')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# 2. Show velocity arrows at each point
ax = axes[1]
ax.plot(path_x, path_y, 'b-', linewidth=1, alpha=0.3)
ax.scatter(*x0, color='green', s=150, zorder=5, marker='*')
ax.scatter(*eps, color='red', s=150, zorder=5, marker='X')

for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    xt = (1 - t) * x0 + t * eps
    # Velocity is constant: v = eps - x0
    ax.annotate('', xy=xt + velocity * 0.15, xytext=xt,
                arrowprops=dict(arrowstyle='->', color='purple', lw=2))

ax.set_title('Velocity Field\n$v = \\epsilon - x_0$ (constant everywhere)')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# 3. Show the reverse (inference) path
ax = axes[2]
# Start from noise, follow negative velocity
x_current = eps.copy()
inference_path = [x_current.copy()]
n_steps = 10
dt = 1.0 / n_steps

for step in range(n_steps):
    # Euler step: x_{t-1} = x_t + v * (sigma_next - sigma_current)
    # Since sigma decreases, delta_sigma is negative, so we subtract velocity
    x_current = x_current - velocity * dt
    inference_path.append(x_current.copy())

inference_path = np.array(inference_path)
ax.plot(inference_path[:, 0], inference_path[:, 1], 'g-o', linewidth=2, markersize=4, label='Inference path')
ax.scatter(*eps, color='red', s=150, zorder=5, marker='X', label='Start (noise)')
ax.scatter(*x0, color='green', s=150, zorder=5, marker='*', label='Goal (clean)')
ax.scatter(*inference_path[-1], color='blue', s=100, zorder=5, marker='D', label='Result')

ax.set_title(f'Inference (Euler, {n_steps} steps)\nFollow velocity in reverse')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.suptitle('Flow Matching in 2D: Forward → Velocity → Reverse', fontsize=14)
plt.tight_layout()
plt.show()

print("Left: Forward process creates a straight line from clean to noisy")
print("Center: The velocity v = ε - x₀ is constant along this line")
print("Right: During inference, we follow the velocity backwards to recover x₀")
print("With enough steps, we arrive exactly at the clean data point!")

### Why Does the Initial Loss ≈ 2.0?

The training loss is the MSE between the predicted velocity and the true velocity. At random initialization:
- The model outputs random predictions (essentially zero useful signal)
- The true velocity is $v = \epsilon - x_0$, where both $\epsilon$ and $x_0$ are standard normal
- $\text{Var}(v) = \text{Var}(\epsilon) + \text{Var}(x_0) = 1 + 1 = 2$

So a random predictor achieves loss ≈ 2.0. As training progresses, the loss should decrease — meaning the model is learning to predict better and better velocities!

A well-trained model on simple data might reach loss ≈ 0.1-0.5, meaning it predicts velocities that are much closer to the ground truth.

In [ ]:
# Training loss on random data (demonstrates the mechanics)
# Using real dimensions: 16 latent channels, 4096-dim text
torch.manual_seed(0)
model_fm = NanoDiT(
    dim=128, in_dim=16, ffn_dim=512, out_dim=16,
    text_dim=4096, freq_dim=64, num_heads=4, num_layers=2,
    patch_size=(1, 2, 2),
)
model_fm.eval()
scheduler = FlowMatchScheduler()

x0 = torch.randn(1, 16, 5, 16, 16)       # Wan VAE latent shape
context = torch.randn(1, 226, 4096)        # T5 embedding shape
loss = scheduler.compute_loss(model_fm, x0, context)

print(f"Flow matching loss (random init): {loss.item():.4f}")
print(f"Expected for random predictor: ~2.0 (since Var(noise - x0) = 2)")
print("With training, this decreases as the model learns the velocity field.")

del model_fm  # free memory

In [ ]:
# Demonstrate a single Euler denoising step
torch.manual_seed(123)
scheduler = FlowMatchScheduler()
scheduler.set_timesteps(num_inference_steps=10, shift=5.0)

print("Sigma schedule (10 steps, shift=5):")
print(f"  Sigmas:    {scheduler.sigmas.numpy().round(4)}")
print(f"  Timesteps: {scheduler.timesteps.numpy().round(1)}")

# Use real latent shape: 16 channels, 5 temporal, 16x16 spatial
x_t = torch.randn(1, 16, 5, 16, 16)
v_pred = torch.randn_like(x_t) * 0.1

sigma_current = scheduler.sigmas[0].item()
sigma_next = scheduler.sigmas[1].item()
delta_sigma = sigma_next - sigma_current

x_t_minus_1 = x_t + v_pred * delta_sigma

print(f"\nEuler step: sigma {sigma_current:.4f} -> {sigma_next:.4f} (delta={delta_sigma:.4f})")
print(f"x_t stats:     mean={x_t.mean().item():.4f}, std={x_t.std().item():.4f}")
print(f"x_{{t-1}} stats: mean={x_t_minus_1.mean().item():.4f}, std={x_t_minus_1.std().item():.4f}")

# Verify against scheduler.step()
x_check = scheduler.step(v_pred, scheduler.timesteps[0], x_t)
print(f"\nManual vs scheduler.step() max diff: {(x_t_minus_1 - x_check).abs().max().item():.2e}")

---
# Section 6: Training

Now we train our NanoDiT on the **DiffSynth-Studio example video dataset** (real video clips with text captions) using the **pretrained Wan 2.1 VAE** (16 latent channels, CausalConv3d) and **umt5-xxl T5 text encoder** (4096-dim embeddings).

The pipeline:
1. Download pretrained VAE and T5 weights (~9.5 GB, cached after first run)
2. Download DiffSynth-Studio example video dataset (~27 MB)
3. Pre-compute T5 text embeddings for all prompts, then free the T5 model (~9 GB)
4. Train NanoDiT with `in_dim=16, out_dim=16, text_dim=4096` (~2.0M params)

Only the NanoDiT trains -- the VAE and T5 are pretrained and frozen. The VAE runs on CPU to save GPU VRAM.

In [ ]:
import os
import gc
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from nano_video_gen.diffusion.flow_match import FlowMatchScheduler
from nano_video_gen.data.dataset import VideoDataset
from nano_video_gen.visualization.viz import plot_training_curves, save_video_grid

# WanVAEWrapper, T5TextEncoder, CachedTextEmbeddings, NanoDiT
# were already imported in earlier cells

### 6.1 Prepare Dataset & Text Embeddings

The DiffSynth-Studio example video dataset was downloaded in the setup cell. Now we create a DataLoader and pre-compute T5 text embeddings for all unique prompts.

In [ ]:
# Verify synthetic dataset (generated in setup cell) and get latent shape
print(f"Dataset directory: {data_dir}")
print(f"Dataset exists: {os.path.exists(data_dir)}")

# Get latent shape from a real video
print(f"\nWan VAE already loaded (latent_channels={vae.latent_channels})")
with torch.no_grad():
    sample_video = demo_dataset[0]["video"].unsqueeze(0)  # [1, 3, 17, 128, 128]
    test_latent = vae.encode(sample_video)
    latent_shape = (1,) + tuple(test_latent.shape[1:])
    print(f"Latent shape: {list(sample_video.shape)} -> {list(test_latent.shape)}")
    del sample_video, test_latent

In [ ]:
# Create dataset and dataloader (128x128, 17 frames)
dataset = VideoDataset(
    base_path=data_dir,
    height=128,
    width=128,
    num_frames=17,
)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)

print(f"Dataset size: {len(dataset)} videos")
print(f"Unique prompts: {dataset.num_prompts}")
print(f"Batches per epoch: {len(dataloader)}")

# Collect unique prompts for T5 encoding
unique_prompts = sorted(set(dataset.prompts))
prompt_to_idx = {p: i for i, p in enumerate(unique_prompts)}
print(f"\nUnique prompts ({len(unique_prompts)}):")
for i, p in enumerate(unique_prompts):
    print(f"  [{i}] {p}")

# Pre-compute T5 text embeddings, then free T5 model (~9 GB)
print("\nLoading T5 text encoder...")
t5 = T5TextEncoder(model_path=model_path, device="cpu")

print("Encoding all prompts...")
all_embeddings = t5.encode(unique_prompts)
print(f"Text embeddings shape: {list(all_embeddings.shape)}")

t5.free_memory()
gc.collect()

# Create cached embeddings on GPU
cached_text = CachedTextEmbeddings(all_embeddings).to(device)
print(f"Cached text embeddings on {device}")

In [ ]:
# Initialize NanoDiT with real dimensions
model = NanoDiT(
    dim=128, in_dim=16, ffn_dim=512, out_dim=16,
    text_dim=4096, freq_dim=64, num_heads=4, num_layers=2,
    patch_size=(1, 2, 2),
).to(device)

total_dit_params = sum(p.numel() for p in model.parameters())
print(f"NanoDiT:  {total_dit_params:>10,} params (~{total_dit_params/1e6:.1f}M)")
print(f"VAE:      pretrained Wan 2.1 (frozen, on CPU)")
print(f"Text:     cached T5 embeddings ({cached_text.num_prompts} prompts x {cached_text.seq_len} tokens x {cached_text.text_dim})")
print(f"Latent:   {list(latent_shape)}")
print(f"Device:   {device}")

### 6.2 Training Loop

Only NanoDiT parameters are trained. The VAE encodes videos on CPU, and T5 embeddings are looked up from the cache.

**Key training decisions explained:**
- **Learning rate (1e-3):** Relatively high for a Transformer, but works because our model is tiny (~2M params). Large models (14B) use much smaller LRs (~1e-5).
- **AdamW optimizer:** The "W" stands for weight decay — it slightly shrinks weights each step, preventing the model from memorizing noise. This is standard for Transformer training.
- **Gradient clipping (max_norm=1.0):** Prevents rare large gradients from destabilizing training. Especially important early in training when the model makes large errors.
- **50 epochs:** With a small dataset, we need many passes. Production models train for much less than 1 epoch over their massive datasets (they see each video at most once).

In [ ]:
# Training configuration
num_epochs = 50 if torch.cuda.is_available() else 15
learning_rate = 1e-3
weight_decay = 0.01
max_grad_norm = 1.0

scheduler = FlowMatchScheduler()
scheduler.set_timesteps(50, shift=5.0)

# Only NanoDiT parameters are trainable
trainable_params = list(model.parameters())
optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)

model.train()
vae.eval()

losses = []
epoch_losses = []

output_dir = '/content/nano-video-gen/outputs'
os.makedirs(output_dir, exist_ok=True)

print(f"Training for {num_epochs} epochs...")
print(f"  Trainable params: {sum(p.numel() for p in trainable_params):,}")
print(f"  VAE: frozen on CPU | Text: cached T5 embeddings on {device}")

for epoch in range(num_epochs):
    epoch_loss_sum = 0.0
    epoch_steps = 0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)

    for batch in progress_bar:
        video = batch["video"].to(device)
        prompt_idx = batch["prompt_idx"].to(device)

        # Look up cached T5 embeddings
        context = cached_text(prompt_idx)

        # Compute flow matching loss (VAE encodes on CPU inside compute_loss)
        loss = scheduler.compute_loss(model, video, context, vae=vae)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, max_grad_norm)
        optimizer.step()

        loss_val = loss.item()
        losses.append(loss_val)
        epoch_loss_sum += loss_val
        epoch_steps += 1
        progress_bar.set_postfix(loss=f"{loss_val:.4f}")

    avg_epoch_loss = epoch_loss_sum / max(epoch_steps, 1)
    epoch_losses.append(avg_epoch_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{num_epochs}: avg_loss={avg_epoch_loss:.4f}")

print(f"\nTraining complete! Final loss: {losses[-1]:.4f}")

In [ ]:
# Loss curve
fig = plot_training_curves(losses, title="Flow Matching Training Loss")
plt.show()

In [ ]:
# Save checkpoint
final_ckpt_path = os.path.join(output_dir, 'checkpoint_final.pt')

torch.save({
    'model': model.state_dict(),
    'text_embeddings': cached_text.embeddings.cpu(),
    'epoch': num_epochs,
    'losses': losses,
    'num_prompts': cached_text.num_prompts,
    'latent_shape': list(latent_shape),
}, final_ckpt_path)

total_dit_params = sum(p.numel() for p in model.parameters())
print(f"Checkpoint saved to: {final_ckpt_path}")
print(f"  NanoDiT params: {total_dit_params:,}")
print(f"  Text embeddings: {list(cached_text.embeddings.shape)}")
print(f"  Latent shape: {list(latent_shape)}")
print(f"  Final loss:   {losses[-1]:.4f}")

---
# Section 7: Inference

Now we use the trained model to generate videos, decoding latents through the Wan 2.1 VAE. We will explore:
1. The full denoising loop with intermediate visualization
2. Classifier-Free Guidance (CFG)
3. Effect of denoising step count
4. Saving generated videos as GIF

In [ ]:
from nano_video_gen.visualization.viz import visualize_denoising_process

# Load checkpoint (in case this section is run independently)
checkpoint_path = '/content/nano-video-gen/outputs/checkpoint_final.pt'

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

    # Real models checkpoint
    latent_shape = tuple(checkpoint['latent_shape'])
    num_prompts = checkpoint['num_prompts']

    model = NanoDiT(
        dim=128, in_dim=16, ffn_dim=512, out_dim=16,
        text_dim=4096, freq_dim=64, num_heads=4, num_layers=2,
        patch_size=(1, 2, 2),
    ).to(device)
    model.load_state_dict(checkpoint['model'])

    cached_text = CachedTextEmbeddings(checkpoint['text_embeddings']).to(device)

    # Re-load WanVAE if not already in memory
    if 'vae' not in dir() or not isinstance(vae, WanVAEWrapper):
        model_path = '/content/nano-video-gen/pretrained_models/Wan2.1'
        vae = WanVAEWrapper(model_path=model_path, device="cpu")

    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
    print(f"Latent shape: {list(latent_shape)}")
else:
    print("Using model from training (already in memory).")

model.eval()
vae.eval()
cached_text.eval()
print("Models set to eval mode.")

### 7.1 Denoising Loop with Visualization

In [ ]:
scheduler = FlowMatchScheduler()
scheduler.set_timesteps(50, shift=5.0)

x = torch.randn(latent_shape, device=device)
ctx = cached_text(torch.tensor([0], device=device))

denoising_frames = []

with torch.no_grad():
    for i, t in enumerate(tqdm(scheduler.timesteps, desc="Denoising (50 steps)")):
        v_pred = model(x, t.unsqueeze(0).to(device), ctx)
        x = scheduler.step(v_pred, t, x)

        if i % 5 == 0:
            decoded = vae.decode(x).cpu()
            denoising_frames.append(decoded)

    final_decoded = vae.decode(x).cpu()
    denoising_frames.append(final_decoded)

print(f"Collected {len(denoising_frames)} snapshots.")
print(f"Final video shape: {final_decoded.shape}")

In [ ]:
# Visualize denoising: first temporal frame
fig = visualize_denoising_process(denoising_frames, frame_idx=0, figsize=(18, 3))
plt.show()

# Middle temporal frame
fig = visualize_denoising_process(denoising_frames, frame_idx=8, figsize=(18, 3))
plt.show()

### 7.2 Classifier-Free Guidance (CFG) — Amplifying the Text Signal

**The Problem:** The model can generate videos that roughly match the prompt, but the connection between text and video can be weak — the model might "play it safe" and generate generic content.

**The Intuition:** Imagine you ask someone to draw "a red sports car." They draw a car that's vaguely reddish. CFG is like saying: *"Look at what you'd draw without any instructions (generic car), then look at what you drew with instructions (reddish car). Now take the DIFFERENCE and amplify it!"*

$$\hat{v}_{\text{guided}} = \underbrace{\hat{v}_{\text{uncond}}}_{\text{generic}} + w \cdot \underbrace{(\hat{v}_{\text{cond}} - \hat{v}_{\text{uncond}})}_{\text{what the text adds}}$$

- $w = 1.0$: Standard generation (no amplification)
- $w = 5.0$ (Wan default): The text-specific component is amplified 5x
- $w > 7.0$: Can become over-saturated (too much amplification distorts the output)

**How it works in practice:** At each denoising step, we run the model **twice**:
1. Once with the real text prompt → $\hat{v}_{\text{cond}}$
2. Once with empty/zero text → $\hat{v}_{\text{uncond}}$

This doubles the compute cost but dramatically improves prompt-following quality. Wan 2.1 uses $w = 5.0$ by default.

In [ ]:
# Compare CFG scales
cfg_scales = [1.0, 3.0, 5.0]
cfg_results = []
seed = 42

ctx_cond = cached_text(torch.tensor([0], device=device))
ctx_uncond = torch.zeros_like(ctx_cond)

scheduler_cfg = FlowMatchScheduler()

for w in cfg_scales:
    torch.manual_seed(seed)
    x = torch.randn(latent_shape, device=device)
    scheduler_cfg.set_timesteps(30, shift=5.0)

    with torch.no_grad():
        for t in scheduler_cfg.timesteps:
            t_batch = t.unsqueeze(0).to(device)
            v_cond = model(x, t_batch, ctx_cond)
            v_uncond = model(x, t_batch, ctx_uncond)
            v_guided = v_uncond + w * (v_cond - v_uncond)
            x = scheduler_cfg.step(v_guided, t, x)
        video = vae.decode(x).cpu()
    cfg_results.append(video)

fig, axes = plt.subplots(1, len(cfg_scales), figsize=(4 * len(cfg_scales), 4))
for i, (w, video) in enumerate(zip(cfg_scales, cfg_results)):
    frame = video[0, :, 0]
    frame = ((frame + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[i].imshow(frame)
    axes[i].set_title(f'CFG scale = {w}')
    axes[i].axis('off')

fig.suptitle('Effect of Classifier-Free Guidance Scale', fontsize=14)
plt.tight_layout()
plt.show()

### 7.3 Effect of Step Count — Quality vs Speed Tradeoff

**More steps = more accurate path from noise to clean video**, but diminishing returns:

- **10 steps:** Coarse approximation — fast but might miss fine details. The Euler integration takes large steps, potentially "overshooting" the optimal path.
- **20 steps:** Good balance for most use cases.
- **50 steps:** High quality — small steps closely follow the true denoising path. Used when quality matters more than speed.

Think of it like GPS navigation:
- 10 steps = "Turn left in 1 mile" → you might overshoot the turn
- 50 steps = "In 200 feet, bear slightly left" → you follow the exact route

In [ ]:
step_counts = [10, 20, 50]
step_results = []
seed = 123
ctx = cached_text(torch.tensor([0], device=device))

for num_steps in step_counts:
    torch.manual_seed(seed)
    x = torch.randn(latent_shape, device=device)

    sched = FlowMatchScheduler()
    sched.set_timesteps(num_steps, shift=5.0)

    with torch.no_grad():
        for t in sched.timesteps:
            v_pred = model(x, t.unsqueeze(0).to(device), ctx)
            x = sched.step(v_pred, t, x)
        video = vae.decode(x).cpu()
    step_results.append(video)
    print(f"  {num_steps:2d} steps done.")

fig, axes = plt.subplots(2, len(step_counts), figsize=(4 * len(step_counts), 7))
for col, (n, video) in enumerate(zip(step_counts, step_results)):
    for row, (frame_idx, frame_label) in enumerate([(0, 'First frame'), (-1, 'Last frame')]):
        frame = video[0, :, frame_idx]
        frame = ((frame + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[row, col].imshow(frame)
        axes[row, col].set_title(f'{n} steps - {frame_label}')
        axes[row, col].axis('off')

fig.suptitle('Effect of Denoising Step Count', fontsize=14)
plt.tight_layout()
plt.show()

### 7.4 Save Generated Videos as GIF

In [ ]:
# Generate videos for multiple prompts and save as GIF
num_to_generate = min(num_prompts, 4)
generated_videos = []

sched = FlowMatchScheduler()
sched.set_timesteps(30, shift=5.0)

for prompt_idx in range(num_to_generate):
    torch.manual_seed(prompt_idx * 7 + 42)
    x = torch.randn(latent_shape, device=device)
    ctx = cached_text(torch.tensor([prompt_idx], device=device))

    with torch.no_grad():
        for t in sched.timesteps:
            v_pred = model(x, t.unsqueeze(0).to(device), ctx)
            x = sched.step(v_pred, t, x)
        video = vae.decode(x).cpu()

    generated_videos.append(video[0])
    print(f"  Generated video for prompt {prompt_idx}")

# Save as GIF grid
gif_path = os.path.join(output_dir, 'generated_videos.gif')
save_video_grid(generated_videos, gif_path, nrow=2, fps=8)
print(f"\nSaved video grid to: {gif_path}")

In [ ]:
# Display middle frame from each generated video
fig, axes = plt.subplots(1, num_to_generate, figsize=(4 * num_to_generate, 4))
if num_to_generate == 1:
    axes = [axes]

for i, video in enumerate(generated_videos):
    mid = video.shape[1] // 2
    frame = video[:, mid]
    frame = ((frame + 1) / 2).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[i].imshow(frame)
    axes[i].set_title(f'Prompt {i} (frame {mid})')
    axes[i].axis('off')

fig.suptitle('Generated Videos -- Middle Frame', fontsize=14)
plt.tight_layout()
plt.show()

print("\nDone! You have trained a miniature video diffusion model and generated videos.")
print("The architecture is identical to Wan 2.1 -- only the scale differs.")

---
## Summary & Key Takeaways

Congratulations! You've built and trained a complete video diffusion model from scratch. Here's what you learned and **why** each piece matters:

### The Three Pillars of Video Generation

| Pillar | Component | Key Insight |
|--------|-----------|-------------|
| **Compression** | Wan 2.1 VAE | Work in latent space (40x smaller) instead of pixel space |
| **Text Understanding** | T5 Encoder | Convert human language into dense vectors the model can use |
| **Denoising** | NanoDiT + Flow Matching | Iteratively remove noise, guided by text, to reveal a video |

### Architecture Decisions & Their Rationale

| Decision | Why? |
|----------|------|
| Latent diffusion (not pixel) | 40x fewer values → feasible compute, semantic features |
| Flow matching (not DDPM) | Simpler math, straight paths, 20-50 steps sufficient |
| Transformer (not U-Net) | Scales better with data/compute, unified architecture |
| 3D RoPE (not absolute pos) | Relative positions, independent axes, generalizes to new sizes |
| AdaIN modulation | One network behaves differently at each noise level |
| CFG guidance | Amplifies text-specific features for better prompt following |

### Scaling: What Changes from Nano to Production?

| Component | Nano | Production | What changes? |
|-----------|------|------------|---------------|
| Model dim | 128 | 5120 | More capacity per token |
| Layers | 2 | 40 | Deeper reasoning |
| DiT Parameters | ~2.0M | ~14B | 7000× more capacity |
| VAE | Wan 2.1 (**same!**) | Wan 2.1 | Nothing — same compression |
| Text encoder | T5 (**same!**) | T5 | Nothing — same text understanding |
| Data | Example clips | Millions of videos | More diverse, longer training |
| Compute | Minutes on 1 GPU | Weeks on 100s of GPUs | Scale, not architecture |

**The key insight:** Scaling is about making the DiT **wider and deeper**, not about changing the architecture. Every building block you studied in this notebook — RoPE, RMSNorm, AdaIN, self-attention, cross-attention, FFN, gating, flow matching — is **exactly the same** in Wan 14B. You now understand the full architecture.